In [2]:
!pip install -U google-genai python-dotenv

     ---------------------------------------- 0.0/56.3 kB ? eta -:--:--
     ------- -------------------------------- 10.2/56.3 kB ? eta -:--:--
     ------- -------------------------------- 10.2/56.3 kB ? eta -:--:--
     ------------- ------------------------ 20.5/56.3 kB 131.3 kB/s eta 0:00:01
     -------------------- ----------------- 30.7/56.3 kB 145.2 kB/s eta 0:00:01
     --------------------------- ---------- 41.0/56.3 kB 178.6 kB/s eta 0:00:01
     ---------------------------------- --- 51.2/56.3 kB 187.0 kB/s eta 0:00:01
     -------------------------------------- 56.3/56.3 kB 184.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.2 MB 2.0 MB/s eta 0:00:01
   -- ------------------------------------- 0.1/1.2 MB 825.8 kB/s eta 0:00:02
   ----- ---------------------------------- 0.2/1.2 MB 1.3 MB/s eta 0:00:01
   ------- -------------------------------- 0.2/1.2 MB 1.2 MB/s eta 0:00:01
   ----

In [4]:
from google import genai
import getpass

print("Google GenAI imported successfully!")

Google GenAI imported successfully!


In [6]:
api_key = getpass.getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=api_key)

print("Gemini client created successfully!")

Enter your Gemini API Key:  ········


Gemini client created successfully!


In [8]:
import os

for file in os.listdir():
    print(file)

.env
.ipynb_checkpoints
data.ipynb
Genai.ipynb
midocean_chunks.json
midocean_chunks_final.json
midocean_documents_cleaned.json
midocean_documents_normalized.json
RQ1_RQ2_final_results.csv
university_guide.json


In [10]:
import json
import pandas as pd

# Load final chunks
with open("midocean_chunks_final.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Number of chunks:", len(chunks))

# Load RQ1 + RQ2 summary
rq1_rq2_results = pd.read_csv("RQ1_RQ2_final_results.csv")

print("\nRQ1/RQ2 results:")
display(rq1_rq2_results)

Number of chunks: 101

RQ1/RQ2 results:


,Method,Recall@5,Recall@10,MRR,NDCG@10
0,BM25 (Sparse),0.80,0.84,0.6455,0.6938
1,BGE-M3 (Dense),0.94,0.96,0.8258,0.8599
2,Hybrid (BM25 + BGE-M3),0.94,0.98,0.7758,0.8257
3,Hybrid + Reranking,0.98,0.98,0.8800,0.9057


In [12]:
import json

with open("data.ipynb", "r", encoding="utf-8") as f:
    notebook = json.load(f)

found = []

for cell in notebook["cells"]:
    source = "".join(cell.get("source", []))

    if "evaluation_queries_final" in source:
        found.append(source)

print("Found", len(found), "cell(s)")

Found 9 cell(s)


In [16]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="اكتب جملة قصيرة تقول فيها مرحباً باللغة العربية."
)

print(response.text)

مرحباً بك، أتمنى لك يوماً جميلاً!


In [18]:
import os

print(os.listdir())

['.env', '.ipynb_checkpoints', 'data.ipynb', 'evaluation_queries_final.json', 'Genai.ipynb', 'midocean_chunks.json', 'midocean_chunks_final.json', 'midocean_documents_cleaned.json', 'midocean_documents_normalized.json', 'RQ1_RQ2_final_results.csv', 'university_guide.json']


In [20]:
import json

# Load the 50 evaluation questions
with open("evaluation_queries_final.json", "r", encoding="utf-8") as f:
    evaluation_queries = json.load(f)

# Load all chunks
with open("midocean_chunks_final.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Number of questions:", len(evaluation_queries))
print("Number of chunks:", len(chunks))

Number of questions: 50
Number of chunks: 101


In [24]:
print(evaluation_queries[0])

{'query_id': 'Q01', 'query': 'ما هي البرامج المتاحة في مرحلة الماجستير؟', 'relevant_chunk_ids': ['DOC_001_CHUNK_0005']}


In [28]:
import json

with open("reranked_all.json", "r", encoding="utf-8") as f:
    reranked_all = json.load(f)

print("Number of queries:", len(reranked_all))
print("First query:", reranked_all[0]["query"])
print("Number of retrieved chunks:", len(reranked_all[0]["results"]))

Number of queries: 50
First query: ما هي البرامج المتاحة في مرحلة الماجستير؟
Number of retrieved chunks: 10


In [34]:
print("Question:")
print(question)

print("\nNumber of retrieved chunks:", len(retrieved_chunks))

print("\nRetrieved chunks:")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"\n--- Chunk {i}: {chunk['chunk_id']} ---")
    print(chunk["text"][:500])

Question:
ما هي البرامج المتاحة في مرحلة الماجستير؟

Number of retrieved chunks: 10

Retrieved chunks:

--- Chunk 1: DOC_001_CHUNK_0005 ---
برامج الماجستير التي تقدمها الجامعة: تقدم الجامعة العديد من برامج الماجستير، وهي كالاتي: 1- ماجستير ادارة الاعمال 2- ماجستير ادارة المشاريع 3- ماجستير ادارة المخاطر 4- ماجستير ادارة الموارد البشرية 5- ماجستير الادارة الصحية 6- ماجستير الاعلام الرقمي 7- ماجستير التسويق الرقمي 8- ماجستير ريادة الاعمال 9- ماجستير الامن السيبراني 10- ماجستير الذكاء الاصطناعي 11- ماجستير القيادة التربوية 12- ماجستير القانون الرقمي 13- ماجستير العلاقات العامة 14- ماجستير التقنية المالية

--- Chunk 2: DOC_003_CHUNK_0061 ---
ما لغة المناهج والشرح؟: برامج تدرس باللغة العربية مع المحتوي باللغة الانجليزية: 1- بكالوريوس الادارة. 2- بكالوريوس ادارة المشاريع. 3- بكالوريوس تقنية المعلومات. 4- ماجستير ادارة الاعمال. 5- ماجستير ادارة المخاطر. 6- ماجستير ادارة المشاريع. 7- ماجستير الامن السيبراني. 8- ماجستير الاعلام الرقمي. 9- ماجستير التسويق الرقمي. 10- ماجستير ريادة الاعمال. 11- م

In [36]:
generation_prompt = """
أنت مساعد أكاديمي لجامعة Midocean.

أجب عن السؤال اعتمادًا فقط على المعلومات الموجودة في المصادر المسترجعة.

السؤال:
{question}

المصادر المسترجعة:
{context}

التعليمات:
- أجب باللغة العربية.
- استخدم المعلومات الموجودة في المصادر فقط.
- لا تضف معلومات من معرفتك العامة.
- إذا لم توجد الإجابة في المصادر، اذكر:
"لا توجد معلومات كافية في المصادر المسترجعة."
- اجعل الإجابة واضحة ومباشرة.
"""

print("Generation prompt created successfully!")

Generation prompt created successfully!


In [38]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [42]:
question = reranked_all[0]["query"]

retrieved_chunks = reranked_all[0]["results"]

context = "\n\n".join(
    [
        f"Chunk ID: {chunk['chunk_id']}\n{chunk['text']}"
        for chunk in retrieved_chunks
    ]
)

generation_prompt = f"""
أنت مساعد أكاديمي لجامعة Midocean.

أجب عن السؤال اعتمادًا فقط على المعلومات الموجودة في المصادر المسترجعة.

السؤال:
{question}

المصادر المسترجعة:
{context}

التعليمات:
- أجب باللغة العربية.
- استخدم المعلومات الموجودة في المصادر فقط.
- لا تضف معلومات من معرفتك العامة.
- إذا لم توجد الإجابة في المصادر، اذكر:
"لا توجد معلومات كافية في المصادر المسترجعة."
- اجعل الإجابة واضحة ومباشرة.
"""

interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input=generation_prompt
)

print("Question:")
print(question)

print("\nGenerated Answer:")
print(interaction.output_text)

Question:
ما هي البرامج المتاحة في مرحلة الماجستير؟

Generated Answer:
بناءً على المصادر المسترجعة، تقدم جامعة ميداوشن برامج الماجستير التالية:

1. ماجستير إدارة الأعمال.
2. ماجستير إدارة المشاريع.
3. ماجستير إدارة المخاطر.
4. ماجستير إدارة الموارد البشرية.
5. ماجستير الإدارة الصحية.
6. ماجستير الإعلام الرقمي.
7. ماجستير التسويق الرقمي.
8. ماجستير ريادة الأعمال.
9. ماجستير الأمن السيبراني.
10. ماجستير الذكاء الاصطناعي.
11. ماجستير القيادة التربوية.
12. ماجستير القانون الرقمي.
13. ماجستير العلاقات العامة.
14. ماجستير التقنية المالية.
15. ماجستير الإدارة العقارية.


In [44]:
generation_prompt = f"""
أنت مساعد أكاديمي لجامعة Midocean.

مهمتك هي الإجابة عن السؤال باستخدام المعلومات الموجودة في المصادر المسترجعة فقط.

السؤال:
{question}

المصادر المسترجعة:
{context}

قواعد الإجابة الإلزامية:
1. لا تستخدم أي معلومة غير موجودة حرفيًا أو بوضوح في المصادر.
2. لا تضف أسماء برامج أو أرقام أو تفاصيل من معرفتك العامة.
3. إذا كانت الإجابة موجودة في أحد المصادر، انقل المعلومات ذات الصلة فقط.
4. إذا كانت المصادر لا تحتوي على معلومات كافية للإجابة، اكتب:
"لا توجد معلومات كافية في المصادر المسترجعة."
5. لا تخمّن ولا تستنتج معلومات غير مذكورة.
6. أجب باللغة العربية بشكل واضح ومباشر.
"""

interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input=generation_prompt
)

print("Question:")
print(question)

print("\nGenerated Answer:")
print(interaction.output_text)

Question:
ما هي البرامج المتاحة في مرحلة الماجستير؟

Generated Answer:
بناءً على المصادر المسترجعة، توفر جامعة ميداوشن برامج الماجستير التالية:

1. ماجستير إدارة الأعمال.
2. ماجستير إدارة المشاريع.
3. ماجستير إدارة المخاطر.
4. ماجستير إدارة الموارد البشرية.
5. ماجستير الإدارة الصحية.
6. ماجستير الإعلام الرقمي.
7. ماجستير التسويق الرقمي.
8. ماجستير ريادة الأعمال.
9. ماجستير الأمن السيبراني.
10. ماجستير الذكاء الاصطناعي.
11. ماجستير القيادة التربوية.
12. ماجستير القانون الرقمي.
13. ماجستير العلاقات العامة.
14. ماجستير التقنية المالية.
15. ماجستير الإدارة العقارية.


In [46]:
test_questions = reranked_all[:3]

for item in test_questions:
    question = item["query"]

    context = "\n\n".join(
        [
            f"Chunk ID: {chunk['chunk_id']}\n{chunk['text']}"
            for chunk in item["results"]
        ]
    )

    prompt = f"""
Answer the question using only the retrieved context.

Question:
{question}

Retrieved Context:
{context}

Instructions:
- Answer in Arabic.
- Do not use external knowledge.
- Do not invent information.
- If the answer is not supported by the retrieved context, say:
"لا توجد معلومات كافية في المصادر المسترجعة."
"""

    interaction = client.interactions.create(
        model="gemini-3-flash-preview",
        input=prompt
    )

    print("=" * 80)
    print("Query ID:", item["query_id"])
    print("Question:", question)
    print("Answer:", interaction.output_text)

Query ID: Q01
Question: ما هي البرامج المتاحة في مرحلة الماجستير؟
Answer: بناءً على السياق المسترجع، توفر الجامعة برامج الماجستير التالية:

1. ماجستير إدارة الأعمال.
2. ماجستير إدارة المشاريع.
3. ماجستير إدارة المخاطر.
4. ماجستير إدارة الموارد البشرية.
5. ماجستير الإدارة الصحية.
6. ماجستير الإعلام الرقمي.
7. ماجستير التسويق الرقمي.
8. ماجستير ريادة الأعمال.
9. ماجستير الأمن السيبراني.
10. ماجستير الذكاء الاصطناعي.
11. ماجستير القيادة التربوية.
12. ماجستير القانون الرقمي.
13. ماجستير العلاقات العامة.
14. ماجستير التقنية المالية.
15. ماجستير الإدارة العقارية.
Query ID: Q02
Question: هل تقدم الجامعة برامج للبكالوريوس؟ وما هي؟
Answer: نعم، تقدم الجامعة العديد من برامج البكالوريوس، وهي كالتالي:

1. بكالوريوس الإدارة.
2. بكالوريوس إدارة المشاريع.
3. بكالوريوس تقنية المعلومات.
4. بكالوريوس الإعلام الرقمي.
5. بكالوريوس التسويق الرقمي.
6. بكالوريوس الريادة والابتكار.
Query ID: Q03
Question: هل توجد برامج دكتوراه في الجامعة؟
Answer: نعم، تقدم الجامعة العديد من برامج الدكتوراه، وهي كالتالي:

1. د

In [48]:
import json
import time

generation_results = []

for i, item in enumerate(reranked_all):

    question = item["query"]

    context = "\n\n".join(
        [
            f"Chunk ID: {chunk['chunk_id']}\n{chunk['text']}"
            for chunk in item["results"]
        ]
    )

    prompt = f"""
Answer the question using only the retrieved context.

Question:
{question}

Retrieved Context:
{context}

Instructions:
- Answer in Arabic.
- Do not use external knowledge.
- Do not invent information.
- If the answer is not supported by the retrieved context, say:
"لا توجد معلومات كافية في المصادر المسترجعة."
"""

    try:
        interaction = client.interactions.create(
            model="gemini-3-flash-preview",
            input=prompt
        )

        answer = interaction.output_text

    except Exception as e:
        answer = f"ERROR: {str(e)}"

    generation_results.append({
        "query_id": item["query_id"],
        "query": question,
        "retrieved_chunk_ids": [
            chunk["chunk_id"] for chunk in item["results"]
        ],
        "answer": answer
    })

    print(f"Completed {i+1}/50 - {item['query_id']}")

    time.sleep(1)

print("\nGeneration completed!")

Completed 1/50 - Q01
Completed 2/50 - Q02
Completed 3/50 - Q03
Completed 4/50 - Q04
Completed 5/50 - Q05
Completed 6/50 - Q06
Completed 7/50 - Q07
Completed 8/50 - Q08
Completed 9/50 - Q09
Completed 10/50 - Q10
Completed 11/50 - Q11
Completed 12/50 - Q12
Completed 13/50 - Q13
Completed 14/50 - Q14
Completed 15/50 - Q15
Completed 16/50 - Q16
Completed 17/50 - Q17
Completed 18/50 - Q18
Completed 19/50 - Q19
Completed 20/50 - Q20
Completed 21/50 - Q21
Completed 22/50 - Q22
Completed 23/50 - Q23
Completed 24/50 - Q24
Completed 25/50 - Q25
Completed 26/50 - Q26
Completed 27/50 - Q27
Completed 28/50 - Q28
Completed 29/50 - Q29
Completed 30/50 - Q30
Completed 31/50 - Q31
Completed 32/50 - Q32
Completed 33/50 - Q33
Completed 34/50 - Q34
Completed 35/50 - Q35
Completed 36/50 - Q36
Completed 37/50 - Q37
Completed 38/50 - Q38
Completed 39/50 - Q39
Completed 40/50 - Q40
Completed 41/50 - Q41
Completed 42/50 - Q42
Completed 43/50 - Q43
Completed 44/50 - Q44
Completed 45/50 - Q45
Completed 46/50 - Q

In [50]:
import json

with open("generation_results.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, ensure_ascii=False, indent=2)

print("Saved successfully!")
print("Number of generated answers:", len(generation_results))

Saved successfully!
Number of generated answers: 50


In [52]:
for item in generation_results[:5]:
    print("=" * 100)
    print("Query ID:", item["query_id"])
    print("Question:", item["query"])
    
    print("\nRetrieved Chunks:")
    print(", ".join(item["retrieved_chunk_ids"]))
    
    print("\nGenerated Answer:")
    print(item["answer"])
    
    print("\n")

Query ID: Q01
Question: ما هي البرامج المتاحة في مرحلة الماجستير؟

Retrieved Chunks:
DOC_001_CHUNK_0005, DOC_003_CHUNK_0061, DOC_003_CHUNK_0056, DOC_001_CHUNK_0006, DOC_001_CHUNK_0003, DOC_001_CHUNK_0016, DOC_003_CHUNK_0060, DOC_003_CHUNK_0054, DOC_001_CHUNK_0018, DOC_001_CHUNK_0004

Generated Answer:
بناءً على السياق المسترجع، البرامج المتاحة في مرحلة الماجستير هي:

1. ماجستير إدارة الأعمال.
2. ماجستير إدارة المشاريع.
3. ماجستير إدارة المخاطر.
4. ماجستير إدارة الموارد البشرية.
5. ماجستير الإدارة الصحية.
6. ماجستير الإعلام الرقمي.
7. ماجستير التسويق الرقمي.
8. ماجستير ريادة الأعمال.
9. ماجستير الأمن السيبراني.
10. ماجستير الذكاء الاصطناعي.
11. ماجستير القيادة التربوية.
12. ماجستير القانون الرقمي.
13. ماجستير العلاقات العامة.
14. ماجستير التقنية المالية.
15. ماجستير الإدارة العقارية.


Query ID: Q02
Question: هل تقدم الجامعة برامج للبكالوريوس؟ وما هي؟

Retrieved Chunks:
DOC_001_CHUNK_0004, DOC_001_CHUNK_0015, DOC_003_CHUNK_0060, DOC_001_CHUNK_0005, DOC_001_CHUNK_0006, DOC_003_CHUNK_0062

In [3]:
print("Hybrid type:", type(hybrid_results))
print("Reranked type:", type(reranked_all))

print("\nFirst Hybrid item:")
print(hybrid_results[0])

print("\nFirst Reranked item:")
print(reranked_all[0])

Hybrid type: <class 'list'>
Reranked type: <class 'list'>

First Hybrid item:
{'query_id': 'Q01', 'query': 'ما هي البرامج المتاحة في مرحلة الماجستير؟', 'relevant_chunk_ids': ['DOC_001_CHUNK_0005'], 'results': [{'rank': 1, 'chunk_id': 'DOC_001_CHUNK_0005', 'document_id': 'DOC_001', 'section': 'برامج الماجستير التي تقدمها الجامعة', 'score': 0.029726775956284153, 'text': 'برامج الماجستير التي تقدمها الجامعة: تقدم الجامعة العديد من برامج الماجستير، وهي كالاتي: 1- ماجستير ادارة الاعمال 2- ماجستير ادارة المشاريع 3- ماجستير ادارة المخاطر 4- ماجستير ادارة الموارد البشرية 5- ماجستير الادارة الصحية 6- ماجستير الاعلام الرقمي 7- ماجستير التسويق الرقمي 8- ماجستير ريادة الاعمال 9- ماجستير الامن السيبراني 10- ماجستير الذكاء الاصطناعي 11- ماجستير القيادة التربوية 12- ماجستير القانون الرقمي 13- ماجستير العلاقات العامة 14- ماجستير التقنية المالية'}, {'rank': 2, 'chunk_id': 'DOC_001_CHUNK_0003', 'document_id': 'DOC_001', 'section': 'مدة الدراسة', 'score': 0.02919863597612958, 'text': 'مدة الدراسة: تتالف 